# Diagnostic: cos_sim_mean Convergence Across Models

Extracts `cos_sim_mean` data from existing notebook cell outputs (no model loading required).
Tests whether Pythia-410m's token fragmentation is a **readout artefact** (dynamics converge
but vocabulary can't name the attractor) or **genuine non-convergence** (dynamics are chaotic).

See `docs/SCALING_ARTEFACT_ANALYSIS.md` for context.

In [ ]:
import json
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

def extract_cos_mean_from_notebook(notebook_path):
    """Parse cos_mean values from notebook cell outputs."""
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    pattern = re.compile(r"iter\s+(\d+):.*cos_mean=([\-\d.]+)")
    
    all_runs = []
    current_run = []
    
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        outputs = cell.get('outputs', [])
        for output in outputs:
            text_lines = output.get('text', [])
            for line in text_lines:
                m = pattern.search(line)
                if m:
                    iteration = int(m.group(1))
                    cos_val = float(m.group(2))
                    if current_run and iteration <= current_run[-1][0]:
                        all_runs.append(current_run)
                        current_run = []
                    current_run.append((iteration, cos_val))
    
    if current_run:
        all_runs.append(current_run)
    
    return all_runs

base = Path('.')
models = {
    'GPT-2 Small (124M)': base / 'gpt2_small' / '01_attractor_dominance.ipynb',
    'GPT-2 Medium (345M)': base / 'gpt2_medium' / '01_attractor_dominance.ipynb',
    'Pythia-160m': base / 'pythia_160m' / '01_attractor_dominance.ipynb',
    'Pythia-410m': base / 'pythia_410m' / '01_attractor_dominance.ipynb',
}

data = {}
for name, path in models.items():
    runs = extract_cos_mean_from_notebook(path)
    data[name] = runs
    print(f"{name}: {len(runs)} prompt runs extracted")

In [ ]:
def aggregate_runs(runs):
    """Compute mean and std of cos_sim_mean at each iteration across all prompts."""
    by_iter = defaultdict(list)
    for run in runs:
        for iteration, cos_val in run:
            by_iter[iteration].append(cos_val)
    
    iters_sorted = sorted(by_iter.keys())
    result = {
        'iterations': iters_sorted,
        'mean': [np.mean(by_iter[i]) for i in iters_sorted],
        'std': [np.std(by_iter[i]) for i in iters_sorted],
        'min': [np.min(by_iter[i]) for i in iters_sorted],
        'max': [np.max(by_iter[i]) for i in iters_sorted],
        'n': [len(by_iter[i]) for i in iters_sorted],
    }
    return result

agg = {}
for name, runs in data.items():
    agg[name] = aggregate_runs(runs)
    iters = agg[name]['iterations']
    means = agg[name]['mean']
    print(f"\n{name} (n={agg[name]['n'][0]} prompts):")
    for i, m, s in zip(iters, means, agg[name]['std']):
        marker = ' <-- converged' if m > 0.999 else ''
        print(f"  iter {i:>4}: mean={m:>7.4f}  std={s:.4f}{marker}")

In [ ]:
import plotly.graph_objects as go

colours = {
    'GPT-2 Small (124M)': '#FF6B6B',
    'GPT-2 Medium (345M)': '#C44D58',
    'Pythia-160m': '#4ECDC4',
    'Pythia-410m': '#556270',
}

fig = go.Figure()

for name, a in agg.items():
    iters = a['iterations']
    means = a['mean']
    stds = a['std']
    upper = [m + s for m, s in zip(means, stds)]
    lower = [m - s for m, s in zip(means, stds)]
    
    fig.add_trace(go.Scatter(
        x=iters + iters[::-1],
        y=upper + lower[::-1],
        fill='toself',
        fillcolor=colours[name].replace(')', ', 0.15)').replace('rgb', 'rgba') if 'rgb' in colours[name] else colours[name] + '26',
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
        name=name,
    ))
    
    fig.add_trace(go.Scatter(
        x=iters,
        y=means,
        mode='lines+markers',
        name=name,
        line=dict(color=colours[name], width=2.5),
        marker=dict(size=6),
    ))

fig.add_hline(y=1.0, line_dash='dash', line_color='grey', opacity=0.5,
              annotation_text='Perfect convergence', annotation_position='bottom left')

fig.update_layout(
    title='cos_sim_mean Convergence Across Models (all 125 prompts)',
    xaxis_title='ATR Iteration',
    yaxis_title='cos_sim_mean (mean across prompts)',
    yaxis_range=[-0.2, 1.1],
    template='plotly_white',
    width=900,
    height=500,
    legend=dict(x=0.02, y=0.98),
)

fig.show()

In [ ]:
# Pythia-410m deep dive: separate converged vs oscillating prompts
p410_runs = data['Pythia-410m']

converged = []
oscillating = []

for run in p410_runs:
    final_cos = run[-1][1] if run else 0
    late_values = [v for i, v in run if i >= 100]
    if late_values and all(v > 0.99 for v in late_values):
        converged.append(run)
    else:
        oscillating.append(run)

print(f"Pythia-410m breakdown:")
print(f"  Converged (cos_mean > 0.99 from iter 100+): {len(converged)}/{len(p410_runs)}")
print(f"  Oscillating: {len(oscillating)}/{len(p410_runs)}")

if converged:
    agg_conv = aggregate_runs(converged)
    print(f"\nConverged subset:")
    for i, m in zip(agg_conv['iterations'], agg_conv['mean']):
        print(f"  iter {i:>4}: mean={m:.4f}")

if oscillating:
    agg_osc = aggregate_runs(oscillating)
    print(f"\nOscillating subset:")
    for i, m, s in zip(agg_osc['iterations'], agg_osc['mean'], agg_osc['std']):
        print(f"  iter {i:>4}: mean={m:>7.4f}  std={s:.4f}")

In [ ]:
# Pythia-410m: overlay individual prompt trajectories
fig2 = go.Figure()

for run in oscillating[:20]:
    iters = [r[0] for r in run]
    vals = [r[1] for r in run]
    fig2.add_trace(go.Scatter(
        x=iters, y=vals, mode='lines',
        line=dict(color='rgba(85,98,112,0.3)', width=1),
        showlegend=False,
    ))

for run in converged[:10]:
    iters = [r[0] for r in run]
    vals = [r[1] for r in run]
    fig2.add_trace(go.Scatter(
        x=iters, y=vals, mode='lines',
        line=dict(color='rgba(78,205,196,0.5)', width=1.5),
        showlegend=False,
    ))

fig2.add_hline(y=1.0, line_dash='dash', line_color='grey', opacity=0.5)

fig2.update_layout(
    title='Pythia-410m: Individual Prompt Trajectories (teal=converged, grey=oscillating)',
    xaxis_title='ATR Iteration',
    yaxis_title='cos_sim_mean',
    yaxis_range=[-0.2, 1.1],
    template='plotly_white',
    width=900,
    height=500,
)

fig2.show()

## Interpretation

**If three models converge (cos_mean → 1.0) and Pythia-410m does not:**
The fragmentation is genuine non-convergence, not a readout artefact. The dynamics are chaotic
at this scale — the operator depth (24 layers) is the primary candidate.

**If Pythia-410m converges but tokens still fragment:**
The attractor exists but the unembedding matrix cannot name it. The readout is the artefact.

**If Pythia-410m shows a mix (some prompts converge, others oscillate):**
The attractor landscape has both shallow and deep basins. Some prompts find stable fixed points;
others orbit in a chaotic region between competing weak attractors. This would be the most
interesting result — it suggests the model's weight geometry has genuine structure that ATR
partially resolves, but the landscape is more complex than smaller models.